In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/erdemyavuz55/drug-preprocessed/preprocessed_test.csv
/kaggle/input/datasets/erdemyavuz55/drug-preprocessed/preprocessed_train.csv


# Importing data

In [2]:
balanced_train = pd.read_csv('/kaggle/input/datasets/erdemyavuz55/drug-preprocessed/preprocessed_train.csv')
drug_test_sample = pd.read_csv('/kaggle/input/datasets/erdemyavuz55/drug-preprocessed/preprocessed_test.csv')

In [3]:
balanced_train.head(3)

,Unnamed: 0,drugName,condition,review,rating,date,usefulCount,tokens
0,206461,Valsartan,Left Ventricular Dysfunction,"""it has no side effect, i take it in combinati...",9.0,"May 20, 2012",27,"['side', 'effect', 'take', 'combination', 'bys..."
1,95260,Guanfacine,ADHD,"""my son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,"['son', 'halfway', 'fourth', 'week', 'intuniv'..."
2,92703,Lybrel,Birth Control,"""i used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,"['used', 'take', 'another', 'oral', 'contracep..."


In [4]:
drug_test_sample.head(3)

,Unnamed: 0,drugName,condition,review,rating,date,usefulCount,tokens
0,163740,Mirtazapine,Depression,"""i've tried a few antidepressants over the yea...",10.0,"February 28, 2012",22,"['tried', 'antidepressant', 'year', 'citalopra..."
1,206473,Mesalamine,"Crohn's Disease, Maintenance","""my son has crohn's disease and has done very ...",8.0,"May 17, 2009",17,"['son', 'crohn', 'disease', 'done', 'well', 'a..."
2,159672,Bactrim,Urinary Tract Infection,"""quick reduction of symptoms""",9.0,"September 29, 2017",3,"['quick', 'reduction', 'symptom']"


# Geleneksel Makine Öğrenimi Modelleri

## Naive Bayes

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# 1. Sentiment etiketlerini oluştur
balanced_train['sentiment'] = (balanced_train['rating'] >= 7).astype(int)

# 2. TF-IDF ile metinleri sayısallaştır
# Tokens sütununu birleştirerek metin formatına dönüştür
balanced_train['text'] = balanced_train['tokens'].apply(lambda x: ' '.join(eval(x)))

# TF-IDF vektörizasyonu
tfidf = TfidfVectorizer(max_features=5000)  # En sık geçen 5000 kelimeyi kullan
X = tfidf.fit_transform(balanced_train['text']).toarray()
y = balanced_train['sentiment']

# 3. Eğitim ve test setlerini ayır
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Naive Bayes Modeli
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 5. Model Performansı
y_pred = nb_model.predict(X_val)
y_pred_probs = nb_model.predict_proba(X_val)[:, 1]

# Doğruluk
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.2f}")

# Classification Report
print("Classification Report:\n", classification_report(y_val, y_pred))

# AUC-ROC
auc_roc = roc_auc_score(y_val, y_pred_probs)
print(f"AUC-ROC: {auc_roc:.2f}")

Validation Accuracy: 0.77
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.43      0.56     10915
           1       0.77      0.95      0.85     21345

    accuracy                           0.77     32260
   macro avg       0.79      0.69      0.71     32260
weighted avg       0.78      0.77      0.75     32260

AUC-ROC: 0.84


## Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Logistic Regression Modeli
log_reg_model = LogisticRegression(max_iter=1000, solver='liblinear')  # Daha büyük veri için 'saga' kullanılabilir
log_reg_model.fit(X_train, y_train)

# Model Performansı
y_pred = log_reg_model.predict(X_val)
y_pred_probs = log_reg_model.predict_proba(X_val)[:, 1]

# Doğruluk
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.2f}")

# Classification Report
print("Classification Report:\n", classification_report(y_val, y_pred))

# AUC-ROC
auc_roc = roc_auc_score(y_val, y_pred_probs)
print(f"AUC-ROC: {auc_roc:.2f}")

Validation Accuracy: 0.82
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.66      0.71     10915
           1       0.84      0.90      0.87     21345

    accuracy                           0.82     32260
   macro avg       0.81      0.78      0.79     32260
weighted avg       0.82      0.82      0.82     32260

AUC-ROC: 0.88


## Random Forest

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)  # Default parameters
rf_model.fit(X_train, y_train)

# Model Predictions
y_pred = rf_model.predict(X_val)
y_pred_probs = rf_model.predict_proba(X_val)[:, 1]

# Model Performance
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.2f}")

# Classification Report
print("Classification Report:\n", classification_report(y_val, y_pred))

# AUC-ROC
auc_roc = roc_auc_score(y_val, y_pred_probs)
print(f"AUC-ROC: {auc_roc:.2f}")

Validation Accuracy: 0.88
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.71      0.80     10915
           1       0.87      0.97      0.92     21345

    accuracy                           0.88     32260
   macro avg       0.89      0.84      0.86     32260
weighted avg       0.89      0.88      0.88     32260

AUC-ROC: 0.95


## Gradient Boosting

In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Gradient Boosting Model
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

# Model Predictions
y_pred = gb_model.predict(X_val)
y_pred_probs = gb_model.predict_proba(X_val)[:, 1]

# Model Performance
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.2f}")

# Classification Report
print("Classification Report:\n", classification_report(y_val, y_pred))

# AUC-ROC
auc_roc = roc_auc_score(y_val, y_pred_probs)
print(f"AUC-ROC: {auc_roc:.2f}")

Validation Accuracy: 0.74
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.30      0.43     10915
           1       0.73      0.96      0.83     21345

    accuracy                           0.74     32260
   macro avg       0.76      0.63      0.63     32260
weighted avg       0.75      0.74      0.69     32260

AUC-ROC: 0.80


# Word Embedding

In [9]:
from gensim.models import Word2Vec

# Tokenları veri setinden al ve listeye dönüştür
train_tokens = balanced_train['tokens'].apply(eval).tolist()

# Word2Vec modelini eğit
word2vec_model = Word2Vec(
    sentences=train_tokens,  # Token listesi
    vector_size=100,         # Embedding vektör boyutu
    window=5,                # Kontekst penceresi boyutu
    min_count=2,             # Minimum kelime frekansı
    workers=4,               # Paralel işleme için işçi sayısı
    sg=0                     # CBOW (0) veya Skip-Gram (1) yöntemi
)

# Modelin eğitildiği kelimelere erişim
vocab = list(word2vec_model.wv.index_to_key)

# Örnek bir kelimenin vektörünü görmek için
example_word = vocab[0]  # İlk kelime
example_vector = word2vec_model.wv[example_word]
print(f"Kelime: {example_word}\nVektör: {example_vector}")

# Embedding'leri kaydetmek için
word2vec_model.save("word2vec_model.model")

# Kaydedilen modeli yüklemek için
# loaded_model = Word2Vec.load("word2vec_model.model")

Kelime: day
Vektör: [-1.136621    1.2274301  -0.5298607   1.5024265  -0.7898611   0.9871688
  0.25901142 -2.2178228  -2.2591033   0.35798976  2.5617228   0.98210645
  0.7357631   0.43274096 -0.09916652 -0.43040884  5.10088     0.7204337
 -2.7622297   1.1482356   2.2718945  -1.7224123   4.25331    -1.2128375
  1.45505     1.5874053   1.032602    2.568745    1.4852504  -0.12370693
 -2.0764143  -2.5125217   1.9202274  -0.68957376  0.26606154 -1.5822582
  2.0111082  -1.190078   -1.3426373   4.353758   -0.5147063   1.0206089
 -1.7910327   1.1382214  -2.6069548   0.15405436  1.1943303  -1.7152605
 -0.20508756  2.510101    0.18918857 -0.5935702  -2.147706    0.32593644
  0.13264404 -1.6517818   1.3974048   0.01046706 -0.40813944  2.151294
  1.3622097   0.15600392 -1.5382664  -0.7731956   2.9312983   1.2607422
 -0.5449293  -0.16612186  0.22676632 -0.63630944 -0.10318385 -0.5349074
  1.7414443   0.77079946 -0.1977281  -0.15235545  1.21461     0.947627
 -0.1269086  -2.2586071  -2.994327    1.520

# Derin Öğrenme Algoritmaları
## LSTM

In [10]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# 1. Sentiment etiketlerini oluştur
# 0-6: Negatif (0), 7-10: Pozitif (1)
balanced_train['sentiment'] = (balanced_train['rating'] >= 7).astype(int)

# 2. Embedding'leri oluştur
def tokens_to_embeddings(tokens, model, vector_size):
    embedding_matrix = []
    for token in tokens:
        if token in model.wv:
            embedding_matrix.append(model.wv[token])
        else:
            embedding_matrix.append([0] * vector_size)  # Bilinmeyen kelimeler için sıfır vektörü
    return embedding_matrix

# Tokenları embedding'e dönüştür ve padding uygula
vector_size = 100
train_embeddings = balanced_train['tokens'].apply(eval).apply(
    lambda x: tokens_to_embeddings(x, word2vec_model, vector_size)
)

max_len = 100  # Sabit dizi uzunluğu
X = pad_sequences(
    [np.array(emb) for emb in train_embeddings], maxlen=max_len, dtype='float32', padding='post', truncating='post'
)
y = balanced_train['sentiment'].values

# Eğitim ve doğrulama setlerini ayır
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. LSTM Modeli
model = Sequential([
    LSTM(128, input_shape=(max_len, vector_size), return_sequences=True),  # Daha derin model
    Dropout(0.3),  # Overfitting'i azaltmak için
    LSTM(64, return_sequences=False),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification için
])

# Daha düşük bir öğrenme oranıyla Adam optimizer
optimizer = Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# 4. EarlyStopping ile eğitim
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Modeli eğit
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20, batch_size=64,
    callbacks=[early_stopping]
)

from sklearn.metrics import classification_report, roc_auc_score

# Model performansı
loss, accuracy = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {accuracy:.2f}")

# Tahminleri al
y_pred_probs = model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int)

# Precision, Recall, F1 Score, ve AUC-ROC
print("Classification Report:\n", classification_report(y_val, y_pred))
auc_roc = roc_auc_score(y_val, y_pred_probs)
print(f"AUC-ROC: {auc_roc:.2f}")

2026-02-25 15:10:00.126435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772032200.531336      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772032200.636953      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772032201.570530      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772032201.570557      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772032201.570560      24 computation_placer.cc:177] computation placer alr

Epoch 1/20


I0000 00:00:1772032261.310947      94 cuda_dnn.cc:529] Loaded cuDNN version 91002


2017/2017 ━━━━━━━━━━━━━━━━━━━━ 33s 14ms/step - accuracy: 0.7582 - loss: 0.5065 - val_accuracy: 0.8098 - val_loss: 0.4140
Epoch 2/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.8222 - loss: 0.4052 - val_accuracy: 0.8248 - val_loss: 0.3940
Epoch 3/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.8372 - loss: 0.3730 - val_accuracy: 0.8386 - val_loss: 0.3631
Epoch 4/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.8541 - loss: 0.3424 - val_accuracy: 0.8462 - val_loss: 0.3586
Epoch 5/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.8679 - loss: 0.3166 - val_accuracy: 0.8506 - val_loss: 0.3441
Epoch 6/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.8795 - loss: 0.2937 - val_accuracy: 0.8439 - val_loss: 0.3511
Epoch 7/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.8887 - loss: 0.2766 - val_accuracy: 0.8448 - val_loss: 0.3529
Epoch 8/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.9003 - loss: 0.25

## CNN

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from keras_tuner import RandomSearch

# Hiperparametre uzayını tanımlamak için bir model fonksiyonu
def build_cnn_model(hp):
    model = Sequential()
    
    # Conv1D Katmanı
    model.add(Conv1D(
        filters=hp.Choice('filters', [64, 128, 256]),  # Filtre sayısı
        kernel_size=hp.Choice('kernel_size', [3, 5, 7]),  # Kernel boyutu
        activation='relu',
        input_shape=(max_len, vector_size)
    ))
    model.add(MaxPooling1D(pool_size=hp.Choice('pool_size', [2, 3])))  # Pooling boyutu
    
    # Ek Katmanlar
    model.add(Flatten())
    model.add(Dense(
        units=hp.Choice('dense_units', [64, 128]),  # Dense katmanı düğüm sayısı
        activation='relu'
    ))
    model.add(Dropout(hp.Choice('dropout', [0.2, 0.3, 0.4])))  # Dropout oranı
    model.add(Dense(1, activation='sigmoid'))  # Çıkış katmanı
    
    # Optimizer ve öğrenme oranı
    learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

# RandomSearch ile Hiperparametre Taraması
tuner = RandomSearch(
    build_cnn_model,
    objective='val_accuracy',  # Optimizasyon amacı
    max_trials=10,  # Denenecek model sayısı
    executions_per_trial=2,  # Her yapılandırma için tekrar sayısı
    directory='cnn_tuning',  # Kayıt dizini
    project_name='cnn_hyperparameter_optimization'
)

# Eğitim seti üzerinde tuner'i çalıştırma
tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,  # Eğitim epoch sayısı
    batch_size=64,
    callbacks=[early_stopping]
)

# En iyi hiperparametreleri al
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

# En iyi modeli al ve eğit
best_model = tuner.hypermodel.build(best_hyperparameters)
best_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[early_stopping]
)

# Modeli değerlendirme
loss, accuracy = best_model.evaluate(X_val, y_val)
print(f"Best CNN Model Validation Accuracy: {accuracy:.2f}")

y_pred_probs = best_model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int)
print("Classification Report:\n", classification_report(y_val, y_pred))
auc_roc = roc_auc_score(y_val, y_pred_probs)
print(f"CNN AUC-ROC: {auc_roc:.2f}")

Trial 10 Complete [00h 02m 24s]
val_accuracy: 0.85959392786026

Best val_accuracy So Far: 0.8644296228885651
Total elapsed time: 00h 26m 42s
Epoch 1/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.6648 - loss: 0.6633 - val_accuracy: 0.8076 - val_loss: 0.4242
Epoch 2/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7888 - loss: 0.4408 - val_accuracy: 0.8230 - val_loss: 0.3963
Epoch 3/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8294 - loss: 0.3763 - val_accuracy: 0.8298 - val_loss: 0.3860
Epoch 4/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8581 - loss: 0.3184 - val_accuracy: 0.8429 - val_loss: 0.3574
Epoch 5/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8766 - loss: 0.2751 - val_accuracy: 0.8491 - val_loss: 0.3586
Epoch 6/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8924 - loss: 0.2398 - val_accuracy: 0.8511 - val_loss: 0.3823
Epoch 7/20
2017/2017 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9058 - los